# WA active-school features for the reviewed model sites

## About the data source

This notebook uses the WA Department of Education **Current Active Schools Semester 1 2026 (DET-024)** workbook. It is a point dataset in WGS84 coordinates with one row per active school. The fixed 2026 Semester 1 snapshot contains 1,152 schools and 19 source columns. Despite the word “Public” in the catalogue title, the workbook contains `GOVERNMENT`, `NON-GOVERNMENT` and `COMMUNITY KINDER` schools, so this notebook describes it as an **active-schools dataset**, not a public-schools-only dataset.

**Access method / API:** the 2026 DET-024 catalogue package currently provides no web-service API. The promoted adapter therefore uses the official Data WA XLSX resource URL over HTTPS only when the integrity-pinned local snapshot is absent. It downloads to a temporary file, verifies the configured SHA-256 digest, and atomically replaces the approved cache path. The older 2025 ArcGIS REST layer is not used because it is a different, outdated annual dataset.

The source tells us where each school is and supplies its school code, name, physical address, education region, sector, remoteness classification, lowest and highest year codes, and reported primary, secondary and total student counts. This information can support measures of school proximity, the type of the nearest school and the year range it serves. It cannot tell us the walking route, school gate location, student travel mode, pedestrian volume, whether students cross the surveyed road, or the amount of school-related pedestrian exposure at that site.

## Notebook and module responsibilities

The reusable implementation now lives in `src/pedflow/context/sources/wa_education_schools.py`. That adapter receives an already validated `sites` DataFrame, performs controlled source acquisition, source validation, cleaning, local-UTM nearest-school calculation, feature construction and source-specific QA, and returns a `FeatureResult`. It does not reopen `model_dataset.xlsx`, assume 49 sites, write integrated outputs, display plots, or store credentials.

This notebook remains the scientific explanation and regression reference for the reviewed 49-site dataset. It reads the sites for review, calls the promoted adapter, and compares the result with `06_outputs/02_feature_extraction/02_05_haoqi_wa_education_features.xlsx`. It does not overwrite that reviewed workbook. Integrated output writing belongs to the shared Stage 2 workflow.

| Output column | Clear meaning | Modelling role |
|---|---|---|
| `site_id` | Stable survey-site key used to join this table back to the modelling data. | Join key only; not a predictor. |
| `education_nearest_active_school_distance_m` | Straight-line planar distance in metres from the survey site to the nearest in-scope active-school point. | The only recommended education predictor in the first model. |
| `education_nearest_school_sector` | Provider-supplied sector of the nearest school: government, non-government or community kinder. | Interpretation and pre-declared sensitivity analysis; not a default first-model predictor. |
| `education_nearest_school_low_year` | Provider-supplied lowest year code of the nearest school, such as `KIN`, `PKG` or `Y07`. | Interpretation and sensitivity analysis; preserve as a category. |
| `education_nearest_school_high_year` | Provider-supplied highest year code of the nearest school, such as `Y06`, `Y10` or `Y12`. | Interpretation and sensitivity analysis; preserve as a category. |
| `education_nearest_school_code` | Official identifier of the matched nearest school. | Audit field only; never a predictor. |
| `education_nearest_school_name` | Official name of the matched nearest school. | Audit and manual checking only; never a predictor. |
| `education_data_status` | Whether the site was successfully matched. Source or contract failures stop this fail-fast workflow. | Data-quality field only; never a substantive predictor. |

The first model should therefore use only the raw distance column from this source. School sector and year range remain available for a separately declared sensitivity check. School identifiers and match status are retained so every result can be traced and checked. Alternative definitions or transformations must be evaluated inside cross-validation rather than selected after viewing all targets.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    # Find the repository root from a notebook or project working directory.
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (
            (path / "pyproject.toml").is_file()
            and (path / "src" / "pedflow").is_dir()
            and (path / "data" / "processed" / "model_dataset.xlsx").is_file()
        ):
            return path
    raise FileNotFoundError(
        f"Could not find the project repository above {current}. Run this notebook inside it."
    )


PROJECT_ROOT = find_project_root()
SOURCE_DIRECTORY = PROJECT_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

from pedflow.context.config import load_context_config
from pedflow.context.sources.wa_education_schools import extract
from pedflow.context.validation import validate_and_normalise_sites

MODEL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "model_dataset.xlsx"
REVIEWED_OUTPUT_PATH = (
    PROJECT_ROOT
    / "06_outputs"
    / "02_feature_extraction"
    / "02_05_haoqi_wa_education_features.xlsx"
)
PIPELINE_CONFIG = load_context_config(PROJECT_ROOT / "configs" / "context_features_v1.json")
SOURCE_CONFIG = next(
    source
    for source in PIPELINE_CONFIG.sources
    if source.name == "wa_education_schools"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Reviewed output: {REVIEWED_OUTPUT_PATH}")
print(f"Shared configuration enabled: {SOURCE_CONFIG.enabled}")

Project root: /workspace
Reviewed output: /workspace/06_outputs/02_feature_extraction/02_05_haoqi_wa_education_features.xlsx
Shared configuration enabled: True


## 1. Read and validate the reviewed site input

The notebook reads the authoritative workbook only to establish the reviewed regression population. It selects the shared four-column site contract: `site_id`, WGS84 `latitude`, WGS84 `longitude`, and `coordinate_crs`. The common validator normalises and checks those columns before the adapter is called.

The 49-row assertion below belongs only to this reviewed notebook. The promoted adapter contains no fixed row-count assumption and is separately tested with two-site fixtures.

In [2]:
model_data = pd.read_excel(
    MODEL_DATASET_PATH,
    sheet_name="Model_Data",
    dtype={"site_id": "string", "coordinate_crs": "string"},
)
sites = validate_and_normalise_sites(model_data)

if len(sites) != 49:
    raise ValueError(f"The reviewed notebook expects 49 sites, found {len(sites)}.")

print(f"Reviewed sites: {len(sites)}")
print(f"Unique site_id values: {sites['site_id'].nunique()}")
print(f"Coordinate reference systems: {sorted(sites['coordinate_crs'].unique())}")

Reviewed sites: 49
Unique site_id values: 49
Coordinate reference systems: ['EPSG:4326']


## 2. Run the promoted adapter and inspect safe provenance

The official [Data WA catalogue record](https://catalogue.data.wa.gov.au/dataset/current-active-schools-semester-1-2026-public-det-024) describes current active school locations for Semester 1 of 2026. It was published by the WA Department of Education, last updated on 13 May 2026, uses GCS WGS 1984 coordinates, and is licensed under [Creative Commons Attribution-NonCommercial 4.0](https://creativecommons.org/licenses/by-nc/4.0/).

The adapter verifies the configured source path and SHA-256 digest before reading the `ActiveSchools` worksheet. It validates the reviewed schema, identifiers, coordinates, sector and year-code vocabularies, then filters records by the configured WA project bounds. The returned metadata contains reproducibility information only: URLs, version, hash, row counts, bounds and method. It contains no credentials or unrestricted responses.

In [3]:
feature_result = extract(sites, SOURCE_CONFIG, PROJECT_ROOT)
features = feature_result.features

print(f"Source name: {feature_result.source_name}")
print(f"Source version: {feature_result.source_version}")
print(f"Source SHA-256: {feature_result.metadata['source_file_sha256']}")
print(f"Retrieval method: {feature_result.metadata['retrieval_method']}")
print(f"Official source rows: {feature_result.metadata['source_row_count']:,}")
print(f"Schools inside project bounds: {feature_result.metadata['in_scope_school_count']:,}")
print(f"Schools outside project bounds: {feature_result.metadata['outside_scope_school_count']:,}")
print("Sector distribution:", feature_result.metadata["sector_counts"])
print("Warnings:")
for warning in feature_result.warnings:
    print(f"- {warning}")

Source name: wa_education_schools
Source version: DET-024-2026-S1
Source SHA-256: ac4257d76e333dcbf2042c7e89b6337fd66b4a990667bc936ed2b1172dd2ffbd
Retrieval method: local_snapshot
Official source rows: 1,152
Schools inside project bounds: 1,150
Schools outside project bounds: 2
Sector distribution: {'GOVERNMENT': 817, 'NON-GOVERNMENT': 317, 'COMMUNITY KINDER': 18}
Warnings:
- Filtered 2 school records outside the configured WA project bounds.
- 2 source records report TotalSchool=0; enrolment fields are not included as features.


## 3. Review the promoted spatial and feature contract

The adapter applies the declared project extent—longitude 112°E to 129°E and latitude 36°S to 13°S—which excludes the two WA-administered schools on Christmas Island and the Cocos (Keeling) Islands. For each supplied site, it estimates the site's WGS84 UTM zone, projects the site and in-scope schools into that local CRS, and selects the smallest Euclidean distance. Ties are resolved deterministically by school code.

The result is a point-to-point planar straight-line distance. It is not walking-network distance, school-gate distance, evidence that students cross the surveyed road, or a school-quality measure. `TotalSchool`, primary/secondary enrolment, remoteness, education region, address and postcode remain excluded for the scientific reasons documented in the reviewed notebook.

Only the approved eight-column table is returned. The module's source-specific validation requires finite non-negative distances, complete matched rows, consistent school code/name pairs, approved categories and one row per supplied `site_id`.

In [4]:
expected_columns = [
    "site_id",
    "education_nearest_active_school_distance_m",
    "education_nearest_school_sector",
    "education_nearest_school_low_year",
    "education_nearest_school_high_year",
    "education_nearest_school_code",
    "education_nearest_school_name",
    "education_data_status",
]

if features.columns.tolist() != expected_columns:
    raise ValueError("The promoted feature columns differ from the reviewed contract.")
if len(features) != len(sites):
    raise ValueError("The promoted feature row count differs from the supplied sites.")
if features["site_id"].tolist() != sites["site_id"].tolist():
    raise ValueError("The promoted adapter changed the supplied site order.")

print(features.head().to_string(index=False))

  site_id  education_nearest_active_school_distance_m education_nearest_school_sector education_nearest_school_low_year education_nearest_school_high_year education_nearest_school_code       education_nearest_school_name education_data_status
250303-01                                20481.040474                  NON-GOVERNMENT                               KIN                                Y06                          1188                  ST JOSEPH'S SCHOOL               matched
250303-02                                 1349.539551                      GOVERNMENT                               KIN                                Y06                          5760       DJIDI DJIDI ABORIGINAL SCHOOL               matched
250303-03                                  318.566350                      GOVERNMENT                               Y07                                Y12                          4014           COLLIE SENIOR HIGH SCHOOL               matched
250303-04                   

## 4. Confirm the reviewed output remains unchanged

The reviewed workbook is a regression reference, not an output target for this notebook. Identifiers, category fields, school names, statuses, row counts and ordering must match exactly. The projected distance uses an absolute tolerance of **0.5 m** with zero relative tolerance. Half a metre is negligible relative to the source coordinate precision but allows harmless numerical differences between supported PROJ releases.

The dry-run one-to-one merge below uses only the supplied site keys and the returned features. It confirms that the promoted table preserves complete coverage without writing the integrated model dataset. The shared Stage 2 I/O layer remains solely responsible for final integrated outputs and manifests.

In [5]:
DISTANCE_ABSOLUTE_TOLERANCE_M = 0.5
reviewed = pd.read_excel(
    REVIEWED_OUTPUT_PATH,
    sheet_name="WA_Education_Features",
    dtype={
        "site_id": "string",
        "education_nearest_school_code": "string",
    },
)

if reviewed.columns.tolist() != features.columns.tolist():
    raise ValueError("The reviewed workbook columns differ from the promoted output.")
if reviewed["site_id"].tolist() != features["site_id"].tolist():
    raise ValueError("The reviewed workbook site order differs from the promoted output.")

exact_columns = [
    column
    for column in features.columns
    if column != "education_nearest_active_school_distance_m"
]
for column in exact_columns:
    pd.testing.assert_series_equal(
        features[column].reset_index(drop=True),
        reviewed[column].reset_index(drop=True),
        check_dtype=False,
        check_names=True,
    )

np.testing.assert_allclose(
    features["education_nearest_active_school_distance_m"],
    reviewed["education_nearest_active_school_distance_m"],
    rtol=0.0,
    atol=DISTANCE_ABSOLUTE_TOLERANCE_M,
)
maximum_distance_difference_m = (
    features["education_nearest_active_school_distance_m"]
    - reviewed["education_nearest_active_school_distance_m"]
).abs().max()

merge_check = sites[["site_id"]].merge(
    features,
    on="site_id",
    how="left",
    validate="one_to_one",
)
if len(merge_check) != len(sites) or merge_check.isna().any().any():
    raise ValueError("The promoted feature table failed the one-to-one merge check.")

print("Reviewed regression comparison: PASS")
print(f"Exact columns checked: {len(exact_columns)}")
print(f"Maximum distance difference: {maximum_distance_difference_m:.9f} m")
print(f"One-to-one merge rows: {len(merge_check)}")

Reviewed regression comparison: PASS
Exact columns checked: 7
Maximum distance difference: 0.000000017 m
One-to-one merge rows: 49


## 5. Quality summary and scientific interpretation

The summary below confirms coverage, distance distribution, nearest-school categories and several pre-declared site checks. The distance distribution matters because a strongly right-skewed variable may need a fold-safe transform or a robust model; it is not a reason to delete a valid remote site.

The scientific conclusion remains unchanged after promotion. Nearest active-school distance is a defensible candidate indicator of proximity to an education destination and local settlement activity, but it is not direct pedestrian exposure. The first model should test only raw distance from this source. Sector and year codes remain interpretation or pre-declared sensitivity fields; school code, name and status remain audit fields. Retention in the deployed feature schema must depend on honest grouped/spatial out-of-fold evaluation, not full-dataset correlation.

In [6]:
distance_summary = features[
    "education_nearest_active_school_distance_m"
].describe(percentiles=[0.25, 0.5, 0.75]).round(1)

print("Match status:")
print(features["education_data_status"].value_counts(dropna=False).to_string())
print("\nNearest-school distance in metres:")
print(distance_summary.to_string())
print("\nNearest-school sector:")
print(features["education_nearest_school_sector"].value_counts().to_string())
print("\nNearest-school low-year code:")
print(features["education_nearest_school_low_year"].value_counts().to_string())
print("\nNearest-school high-year code:")
print(features["education_nearest_school_high_year"].value_counts().to_string())
print("\nDistinct nearest schools:", features["education_nearest_school_code"].nunique())

spot_check_ids = ["250303-15", "250303-10", "250341-05", "250341-02"]
spot_checks = features.loc[features["site_id"].isin(spot_check_ids)].copy()
spot_checks["site_id"] = pd.Categorical(
    spot_checks["site_id"], categories=spot_check_ids, ordered=True
)
spot_checks = spot_checks.sort_values("site_id")
print("\nPre-declared spot checks:")
print(
    spot_checks[
        [
            "site_id",
            "education_nearest_school_name",
            "education_nearest_school_sector",
            "education_nearest_active_school_distance_m",
        ]
    ].to_string(index=False)
)

Match status:
education_data_status
matched    49

Nearest-school distance in metres:
count       49.0
mean      1075.4
std       2867.1
min         58.9
25%        340.7
50%        514.6
75%        920.4
max      20481.0

Nearest-school sector:
education_nearest_school_sector
GOVERNMENT        31
NON-GOVERNMENT    18

Nearest-school low-year code:
education_nearest_school_low_year
KIN    35
PKG     8
Y07     4
Y11     1
Y10     1

Nearest-school high-year code:
education_nearest_school_high_year
Y06    32
Y12    13
Y10     4

Distinct nearest schools: 46

Pre-declared spot checks:
  site_id education_nearest_school_name education_nearest_school_sector  education_nearest_active_school_distance_m
250303-15    GREENFIELDS PRIMARY SCHOOL                      GOVERNMENT                                  964.017512
250303-10            ST BRIGID'S SCHOOL                  NON-GOVERNMENT                                  373.590662
250341-05   BAYNTON WEST PRIMARY SCHOOL                      GO